In [1]:
import os
import requests
import json
import logging
from tqdm import tqdm
from sqlalchemy import text
from contextlib import contextmanager

from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

ImportError: cannot import name 'TypeAliasType' from 'typing_extensions' (C:\Users\bdrex\anaconda3\lib\site-packages\typing_extensions.py)

In [ ]:
class SQLAlchemyClient:
    def __init__(self, isolation_level=None, sent_to_local: bool = False):
        database_host = os.getenv("POSTGRES_DB_HOST", "localhost")
        database_port = os.getenv("POSTGRES_DB_PORT", 5431)
        database_user = os.getenv("POSTGRES_DB_USER", "postgres")
        database_pass = os.getenv("POSTGRES_DB_PASS", "postgres")
        database_name = os.getenv("POSTGRES_DB_NAME", "newsaggregator")
        if sent_to_local:
            database_host = "localhost"
            database_port = 5431
            database_user = "postgres"
            database_pass = "postgres"
            database_name = "newsaggregator"

        self.database_uri = f"postgresql+psycopg2://{database_user}:{database_pass}@{database_host}:{database_port}/{database_name}"

        engine = create_engine(self.database_uri, isolation_level=isolation_level)
        self.Session = sessionmaker(bind=engine)

    @contextmanager
    def get_session(self):
        session = self.Session()

        try:
            yield session

        except Exception as e:  # noqa E722
            logger.error(
                f"Error occurred when accessing database using SQLAlchemy. Rolling back...\nException: {e}"
            )
            session.rollback()
        finally:
            session.close()


In [ ]:
sql_alchemy_client = SQLAlchemyClient()

In [ ]:
target_news = []
with sql_alchemy_client.get_session() as session:
    result = session.execute(
        text(
            f"SELECT sitemap_id, headline, sources, category"
            f"  FROM sitemaps as a"
            f" LIMIT 10000"
        )
    )

    for result in result:
        sitemap_id = result[0]
        headline = result[1]
        sources = result[2]
        category = result[3]
        target_news.append({"sitemap_id": sitemap_id, "headline": headline, "sources": sources, "category": category})

In [ ]:
def create_prompt(article):
    prompt = f"""Klasifikasikan judul artikel berita berikut ke dalam satu kategori yang paling sesuai:

    Politik, Ekonomi/Bisnis, Teknologi, Sains, Olahraga, Hiburan, Lingkungan, Pendidikan, Gaya Hidup, Kesehatan, atau Lainnya
    
    Judul artikel: {article}

    Berikan hanya nama kategori sebagai jawaban, tanpa penjelasan tambahan.

    Contoh output:
    Politik
    """
    return prompt

In [ ]:
# Set the headers for the POST request
# Send the POST request

In [ ]:
for target_new in target_news:
    prompt = create_prompt(target_new['headline'])
    headers = {'Content-Type': 'application/json'}
    sent_json = {"model": "llama3.1",
                 "messages": [{ "role": "user", "content": prompt}],
                 "stream": False
                }
    response = requests.post('http://localhost:11434/api/chat', 
                             json=sent_json, 
                             headers=headers)
    if response.status_code == 200:
        new_category = json.loads(response.content)['message']['content']
    target_new['new_category'] = category

In [2]:
import pandas as pd

In [7]:
target_news = pd.read_csv('target_news.csv', encoding='latin1')

In [12]:
target_news.loc[target_news.new_category == 'Kematian']

,sitemap_id,headline,sources,category,new_category
2467,3867615,"Satu Keluarga Tewas di Malang, Wasiat Orang Tu...",INEWS,news,Kematian


In [44]:
target_news.new_category = [x.split("\n")[0].strip().replace(".",'') for x in target_news.new_category]

In [54]:
target_news.loc[target_news.new_category == 'Ibadah']

,sitemap_id,headline,sources,category,new_category
2472,3867620,Jadwal Shalat & Imsak,INEWS,news,Ibadah
3613,3868743,"Sholat Jumat Perdana di Masjidil Haram, Jamaah...",OKEZONE,news,Ibadah


In [58]:
new_new_category = []
for x in target_news.new_category:
    if x in ['Ibadah', 'Islam', 'Kisah Perjuangan Penjual Karak Berhasil Naik Haji']:
        new_new_category.append("Agama")
    elif x in ['Korupsi', 'Kriminil', 'Kriminal/Kejahatan', 'Narkotika', 'Kriminalitas']:
        new_new_category.append("Kriminal")
    else:
        new_new_category.append(x)

In [62]:
target_news['new_category'] = new_new_category

In [79]:
selected_category = target_news.new_category.value_counts().iloc[:13].keys()

target_news = target_news.loc[target_news.new_category.isin(selected_category)]

In [83]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import numpy as np

# Assuming you have your data in X (text) and y (labels)
# X should be a list of strings (your text data)
# y should be a list of integers from 0 to 12 (for 13 categories)

# Hyperparameters
max_words = 10000  # Maximum number of words in the vocabulary
max_len = 200  # Maximum length of each text sequence
embedding_dim = 100  # Dimensionality of embedding layer
lstm_hidden_size = 128  # Number of LSTM hidden units
num_classes = 13  # Number of classification categories

# Tokenization and vocabulary building
tokenizer = get_tokenizer("basic_english")
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(X), max_tokens=max_words)
vocab.set_default_index(0)  # Set OOV token

# Text to index conversion
text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: x

# Custom dataset
class TextClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        text = text_pipeline(self.X[idx])
        label = label_pipeline(self.y[idx])
        return torch.tensor(text), torch.tensor(label)

# Padding function
def collate_batch(batch):
    label_list, text_list = [], []
    for (_text, _label) in batch:
        label_list.append(_label)
        processed_text = _text[:max_len] if len(_text) > max_len else _text
        text_list.append(torch.cat([processed_text, torch.zeros(max_len - len(processed_text))]))
    return torch.stack(text_list).to(torch.int64), torch.tensor(label_list)

# Model definition
class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_hidden_size, num_classes):
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden_size, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(lstm_hidden_size * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out[:, -1, :]  # Take the last time step output
        fc1_out = self.fc1(lstm_out)
        relu_out = self.relu(fc1_out)
        dropout_out = self.dropout(relu_out)
        output = self.fc2(dropout_out)
        return output

# Initialize model, loss function, and optimizer
model = TextClassificationModel(len(vocab), embedding_dim, lstm_hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Create dataset and dataloader
dataset = TextClassificationDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)

# Training loop
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_text, batch_labels in dataloader:
        batch_text, batch_labels = batch_text.to(device), batch_labels.to(device)
        
        optimizer.zero_grad()
        output = model(batch_text)
        loss = criterion(output, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}")

# Function to predict category for new text
def predict_category(text):
    model.eval()
    with torch.no_grad():
        text_tensor = torch.tensor(text_pipeline(text)).unsqueeze(0).to(device)
        output = model(text_tensor)
        return output.argmax(dim=1).item()

# Example usage
# new_text = "Your new text here"
# predicted_category = predict_category(new_text)
# print(f"Predicted category: {predicted_category}")

13